# Complexity & figures — Ancilla-mediated fixed-point quantum search

This notebook reproduces every figure in the paper from the **corrected schedule**.

**Model.** With $N$ items and $M$ solutions, $\theta=\arcsin\sqrt{M/N}$. After each
failed fixed-point check the state resets to angle $2\theta$; the $j$-th check
($j=0,1,\dots,a-1$) is applied at the geometrically doubling angle $2^{j}\theta$,
reached from the reset by $g_j$ Grover operations with $g_0=0,\ g_j=2^{j-1}-1$.
Hence the per-check oracle cost is $o_j=1+g_j$ and the **cumulative** cost to reach
check $j$ is $O_j=\sum_{l\le j}o_l=2^{j}$.

**Correction over the original notebook.** The earlier `calculate_exp` used the
cost $2^l-l$ and weighted it by the *cumulative* success probability; the
expectation must weight $O_l=2^l$ by the probability of *first* success at round
$l$. With both fixed, $\mathbb{E}(O_j)/k_\mathrm{opt}\approx1.32$, i.e.
$\mathbb{E}(O_j)\sim\sqrt{N/M}$ — matching the manuscript. The probability
computation was already correct.

In [ ]:
import numpy as np
from math import sqrt, pi, asin, sin, cos
import matplotlib as mpl
import matplotlib.pyplot as plt

USETEX = False   # True on a machine with latex+dvipng to match the paper's CM text exactly
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.family": "serif", "text.usetex": USETEX, "mathtext.fontset": "cm",
    "font.size": 8.5, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 7.4, "axes.linewidth": 0.7, "lines.linewidth": 1.4,
    "xtick.direction": "in", "ytick.direction": "in",
    "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "0.7",
    "grid.linewidth": 0.4, "grid.alpha": 0.5,
})
BLUE, VERM, GREEN, GREY = "#0072B2", "#D55E00", "#009E73", "#555555"  # Okabe-Ito
COL = (3.4, 2.55)
N = 10**10

## Core quantities: success probability and expected oracle calls

In [ ]:
def calculate_CP(theta):
    """Total probability the doubling schedule converges (checks at 2^l theta)."""
    l, p, fp = -1, 0.0, 1.0
    while theta * 2**l < pi/2:
        l += 1
        p  += fp * sin(2**l * theta)**2
        fp *= cos(2**l * theta)**2
    return p

def calculate_exp(theta):
    """Expected oracle calls, corrected: O_l = 2^l weighted by P(first success at l)."""
    l, fp, E, lastO = -1, 1.0, 0.0, 1
    while theta * 2**l < pi/2:
        l += 1
        O_l  = 2**l
        E   += O_l * (sin(2**l * theta)**2) * fp   # cost * P(first success exactly at l)
        fp  *= cos(2**l * theta)**2
        lastO = O_l
    E += fp * lastO                                # never-converged tail pays full budget
    return E

## Sweep over $M$ (dense for small $M$, log-spaced tail)

In [ ]:
M = np.array(list(range(1, 50)) +
             list(np.unique(np.logspace(np.log10(50), np.log10(N//100), 1400).astype(np.int64))))
theta = np.array([asin(sqrt(m/N)) for m in M])
kopt  = np.array([int(pi/4/t - 0.5) for t in theta])   # Grover optimal iterations (known M)
P     = np.array([calculate_CP(t)  for t in theta])
E     = np.array([calculate_exp(t) for t in theta])
worst = pi * np.sqrt(N / M)                            # analytic worst-case guarantee
print("min P (plotted) = %.4f;  E/kopt mean = %.3f, max = %.3f" % (P.min(), (E/kopt).mean(), (E/kopt).max()))

## Fig. 4 — Probability to converge vs $M$

In [ ]:
fig, ax = plt.subplots(figsize=COL)
ax.set_xscale('log'); ax.grid(True, which='both', ls='--')
ax.plot(M, P, color=BLUE, lw=1.4)
ax.axhline(0.9259, color=VERM, ls=(0,(5,2)), lw=1.1)
ax.text(0.98, 0.9259, r'analytic bound $0.9259$', color=VERM, fontsize=7,
        ha='right', va='bottom', transform=ax.get_yaxis_transform())
ax.set_xlabel(r'Number of solutions $M$'); ax.set_ylabel(r'Probability to converge')
ax.set_ylim(0.90, 1.005); ax.set_xlim(1, N//100)
fig.tight_layout(pad=0.3); fig.savefig('fig4_probability.pdf'); plt.show()

## Fig. 5 — Expected oracle calls vs $M$ (corrected)

In [ ]:
fig, ax = plt.subplots(figsize=COL)
ax.set_xscale('log'); ax.set_yscale('log'); ax.grid(True, which='both', ls='--')
ax.plot(M, worst, color=GREY, ls=(0,(2,2)), lw=1.0, label=r'Worst case $\pi\sqrt{N/M}$')
ax.plot(M, E,     color=VERM, lw=1.5, label=r'Fixed-point (expected)')
ax.plot(M, kopt,  color=BLUE, lw=1.5, label=r'Grover optimal (known $M$)')
ax.set_xlabel(r'Number of solutions $M$'); ax.set_ylabel(r'Oracle calls')
ax.set_xlim(1, N//100); ax.legend(loc='upper right')
fig.tight_layout(pad=0.3); fig.savefig('fig5_oraclecalls.pdf'); plt.show()

## Fig. A (new) — the soufflé problem

Standard Grover's single-shot success $\sin^2((2k+1)\theta)$ **oscillates**: stop a
little late and it collapses. The ancilla fixed-point scheme accumulates success
across checkpoints and **never loses progress**.

In [ ]:
th = 0.075; kmax = int(2.4*(pi/4/th)); ks = np.arange(0, kmax+1)
Pg = np.sin((2*ks+1)*th)**2
fp, p, l, Oj, Pj = 1.0, 0.0, -1, [], []
while th*2**l < pi/2:
    l += 1; p += fp*sin(2**l*th)**2; fp *= cos(2**l*th)**2; Oj.append(2**l); Pj.append(p)
Oj, Pj = np.array(Oj), np.array(Pj)
kpk = int(round(pi/4/th-0.5)); ktr = int(round((pi/th-1)/2))
fig, ax = plt.subplots(figsize=(3.5,2.6)); ax.grid(True, ls='--')
ax.plot(ks, Pg, color=BLUE, lw=1.3, label=r'Standard Grover $\sin^2((2k+1)\theta)$')
ax.step(np.r_[0,Oj], np.r_[0,Pj], where='post', color=VERM, lw=1.6, label=r'Fixed-point (cumulative)')
ax.scatter(Oj, Pj, s=11, color=VERM, zorder=6)
ax.scatter([kpk],[np.sin((2*kpk+1)*th)**2], s=16, color=BLUE, zorder=6)
ax.axhline(0.9259, color=GREY, ls=(0,(4,2)), lw=0.9)
ax.text(kmax*0.985, 0.9259, r'$0.9259$', color=GREY, fontsize=7, ha='right', va='bottom')
ax.annotate('overshoot', xy=(ktr, np.sin((2*ktr+1)*th)**2), xytext=(ktr-1, 0.30),
            fontsize=7.2, color=BLUE, ha='center', arrowprops=dict(arrowstyle='->', color=BLUE, lw=0.8))
ax.set_xlabel(r'Oracle calls'); ax.set_ylabel(r'Success probability')
ax.set_xlim(0,kmax); ax.set_ylim(0,1.03); ax.legend(loc='center right')
fig.tight_layout(pad=0.3); fig.savefig('figA_souffle.pdf'); plt.show()

## Fig. B (new) — the worst-case bound $p_{c,1}(x)=1-\cos^2x\cos^2 2x$

Justifies the headline number: the two-index convergence probability never drops
below $0.9259$ on $[\pi/4,\pi/2)$, attained at $x\approx1.150$.

In [ ]:
x = np.linspace(pi/4, pi/2, 800); pc1 = 1 - np.cos(x)**2*np.cos(2*x)**2
xm = 1.15026; pm = 1 - cos(xm)**2*cos(2*xm)**2
fig, ax = plt.subplots(figsize=COL); ax.grid(True, ls='--')
ax.plot(x, pc1, color=BLUE, lw=1.6); ax.scatter([xm],[pm], s=16, color=VERM, zorder=6)
ax.annotate(r'min $=0.9259$'+'\n'+r'at $x\approx1.150$', xy=(xm,pm), xytext=(1.18,0.955),
            fontsize=7.2, color=VERM, arrowprops=dict(arrowstyle='->', color=VERM, lw=0.8))
ax.set_xlabel(r'$x=2^{c}\theta$'); ax.set_ylabel(r'$p_{c,1}(x)=1-\cos^2 x\,\cos^2 2x$')
ax.set_xlim(pi/4, pi/2); ax.set_ylim(0.92, 1.005)
ax.set_xticks([pi/4, 3*pi/8, pi/2]); ax.set_xticklabels([r'$\pi/4$', r'$3\pi/8$', r'$\pi/2$'])
fig.tight_layout(pad=0.3); fig.savefig('figB_worstcase.pdf'); plt.show()

## Fig. C (new, optional) — how the doubling schedule converges

Cumulative success $P_j$ vs checkpoint index for three problem sizes; larger $M$
(bigger $\theta$) needs fewer doublings.

In [ ]:
fig, ax = plt.subplots(figsize=(3.5,2.6)); ax.grid(True, ls='--')
for m, c, lab in [(1,BLUE,r'$M=1$'),(100,GREEN,r'$M=10^{2}$'),(10**4,VERM,r'$M=10^{4}$')]:
    t = asin(sqrt(m/N)); fp, p, l, xs, ys = 1.0, 0.0, -1, [], []
    while t*2**l < pi/2:
        l += 1; p += fp*sin(2**l*t)**2; fp *= cos(2**l*t)**2; xs.append(l); ys.append(p)
    ax.plot(xs, ys, marker='o', ms=3, color=c, lw=1.2, label=lab)
ax.axhline(0.9259, color=GREY, ls=(0,(4,2)), lw=0.9)
ax.text(0.5, 0.9259, r'$0.9259$', color=GREY, fontsize=7, va='bottom')
ax.set_xlabel(r'Checkpoint index $j$ (angle $2^{j}\theta$)'); ax.set_ylabel(r'Cumulative success $P_j$')
ax.set_ylim(0,1.03); ax.legend(loc='lower right')
fig.tight_layout(pad=0.3); fig.savefig('figC_cumulative.pdf'); plt.show()